### Import Libaries

In [42]:
from serpapi import GoogleSearch
from datetime import date,timedelta,datetime
import numpy as np
import pandas as pd
import time
import sys
pd.set_option('display.max_columns', None) # Used for Pandas, make hidden columns/rows visiable

### Load Dataset

In [ ]:
airports = pd.read_csv('major_international_airports_clean.csv').iloc[:-4]
display(airports)

,IATA_Code,Airport_Name,City,Country
0,DUB,Dublin Airport,Dublin,Ireland
1,AKL,Auckland Airport,Auckland,New Zealand
2,YYZ,Toronto Pearson International Airport,Toronto,Canada
3,YVR,Vancouver International Airport,Vancouver,Canada
4,YUL,Montréal-Trudeau International Airport,Montreal,Canada
5,YEG,Edmonto International Airport,Edmonto,Canada
6,YYC,Calgary International Airport,Calgary,Canada
7,JFK,John F. Kennedy International Airport,New York,USA
8,LGA,LaGuardia Airport,New York,USA
9,EWR,Newark Liberty International Airport,Newark,USA


In [44]:
print(*airports['IATA_Code'].values,sep=',')

DUB,AKL,YYZ,YVR,YUL,YEG,YYC,JFK,LGA,EWR,LAX,SFO,ORD,DFW,ATL,MIA,CDG,ORY,LHR,LGW,AMS,FRA,MUC,MAD,BCN,FCO,MXP,DXB,DOH,HND,NRT,ICN,PEK,PVG,SIN,BKK,SYD,MEL,DEL,BOM


### Set Up DEP DES list

In [ ]:
# Create Dep and Des lists, Slices of Departures
Des = ['DUB','AKL']
Dep = airports['IATA_Code'].to_numpy().reshape(4,10).tolist()
Des,Dep 


(['DUB', 'AKL'],
 [['DUB', 'AKL', 'YYZ', 'YVR', 'YUL', 'YEG', 'YYC', 'JFK', 'LGA', 'EWR'],
  ['LAX', 'SFO', 'ORD', 'DFW', 'ATL', 'MIA', 'CDG', 'ORY', 'LHR', 'LGW'],
  ['AMS', 'FRA', 'MUC', 'MAD', 'BCN', 'FCO', 'MXP', 'DXB', 'DOH', 'HND'],
  ['NRT', 'ICN', 'PEK', 'PVG', 'SIN', 'BKK', 'SYD', 'MEL', 'DEL', 'BOM']])

### Flight Data Extractor

In [5]:
# Fligh Catalog Datset
# Airport depature & destination name, IATA airport code, date, and price, ticket sold_by   
flghDta = pd.DataFrame(columns=[
        'apt_name_dp','apt_IATA_dp','apt_time_dt_dp',
        'apt_name_ds','apt_IATA_ds','apt_time_dt_ds',
        'price','ticket_sold_by'])

def dataFightExt(data): # Process data for 
    for fd in data:
        fligths = fd['flights'][0]
      
        dp,ds= fligths['departure_airport'],fligths['arrival_airport']
        dp_name,ds_name = dp['name'],ds['name']
        dp_IATA,ds_IATA = dp['id'],ds['id'] 

        dp_dttm,ds_dttm = dp['time'],ds['time']
        dp_dttm_obj = datetime.strptime(dp_dttm,'%Y-%m-%d %H:%M')
        ds_dttm_obj = datetime.strptime(ds_dttm,'%Y-%m-%d %H:%M')
        dp_dt = dp_dttm_obj.date()
        ds_dt = ds_dttm_obj.date()

        try:
            tck_sld_by = fligths['ticket_also_sold_by']
        except:
            tck_sld_by = []

        price = fd['price']

        flghDta.loc[len(flghDta)] = [
            dp_name,dp_IATA,dp_dt,
            ds_name,ds_IATA,ds_dt,
            price,tck_sld_by]

### SerpAPI Future Flight Data

In [ ]:
i = -1 # When issuess happen set this to the last _ here to resume, purge will happen to final dataset to remove any fligths found with this dep list to start again properly 
if i >= 0:# Purge 
    flghDta = flghDta[~flghDta['apt_IATA_dp'].isin(Dep[i])]

outbound = date(2026,3,12) # Flight time range, start date manual change, depends on when the next time this code is run  
EndPoint = outbound + timedelta(days=270)

Limit = False #Used for monthly usage control 
str_des = ','.join(Des)
for _,d in enumerate(Dep): # Take a row of dep for search with des's, find all flight's and save into flghdata dataframe
    if _ < i: pass # this will be used when we want to start at error point again
    str_d = ','.join(d)
    while outbound <= EndPoint: # go through every 2 days grabing flight paths
        print(f"{_} || {outbound}//{EndPoint} || DEP's: {d} || DES: {Des} || FLgD: {len(flghDta)}", end='\r') # This helps with knowing what dep set we are on
        params = {
            "api_key": "API_KEY", 
            "engine": "google_flights",
            "hl": "en",
            "gl": "ca",
            "departure_id": str_d,
            "arrival_id": str_des,
            "outbound_date": str(outbound),
            "currency": "CAD",
            "sort_by": "3",
            "type": "2",
            "stops": "1"
        }

        chkact = GoogleSearch(params).get_account()
        if chkact['this_hour_searches'] > 189: # Limiter for hour
            time.sleep(3600)
        if chkact['this_month_usage'] > 989: # Limiter for month
            Limit = True
            break
    
        search = GoogleSearch(params)
        results = search.get_dict()

        try:
            dataFightExt(results['other_flights'])
        except Exception as e: # Happens when other_flights doesn't exist
            pass

        outbound += timedelta(days=2) # move to next time point

        sys.stdout.flush()
        time.sleep(0.1)
    
    if Limit == True: break

    outbound = date(2026,3,12) # reset start date for the next dep

flghDta = flghDta.convert_dtypes()
display(flghDta.head(5))

,apt_name_dp,apt_IATA_dp,apt_time_dt_dp,apt_name_ds,apt_IATA_ds,apt_time_dt_ds,price,ticket_sold_by
0,John F. Kennedy International Airport,JFK,2026-03-12,Dublin Airport,DUB,2026-03-13,1739,[]
1,Toronto Pearson International Airport,YYZ,2026-03-12,Dublin Airport,DUB,2026-03-13,1483,[]
2,Toronto Pearson International Airport,YYZ,2026-03-12,Dublin Airport,DUB,2026-03-13,7232,[Lufthansa]
3,John F. Kennedy International Airport,JFK,2026-03-12,Auckland Airport,AKL,2026-03-14,2264,[United]
4,Newark Liberty International Airport,EWR,2026-03-12,Dublin Airport,DUB,2026-03-13,1844,"[Lufthansa, Austrian, Brussels Airlines]"


In [28]:
flghDta['ticket_sold_by'] = flghDta['ticket_sold_by'].apply(lambda x: ','.join(x))

In [29]:
flghDta.info()

<class 'pandas.core.frame.DataFrame'>
Index: 14566 entries, 0 to 14565
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   apt_name_dp     14566 non-null  string
 1   apt_IATA_dp     14566 non-null  string
 2   apt_time_dt_dp  14566 non-null  object
 3   apt_name_ds     14566 non-null  string
 4   apt_IATA_ds     14566 non-null  string
 5   apt_time_dt_ds  14566 non-null  object
 6   price           14566 non-null  Int64 
 7   ticket_sold_by  14566 non-null  object
dtypes: Int64(1), object(3), string(4)
memory usage: 1.0+ MB


In [34]:
flghDta2 = flghDta.sort_values(by=['apt_IATA_dp','apt_IATA_ds','apt_time_dt_ds','price'],ignore_index=True).drop_duplicates(subset=['apt_IATA_dp','apt_IATA_ds','apt_time_dt_ds','price','ticket_sold_by'],ignore_index=True)
flghDta2.shape

(10773, 8)

In [35]:
flghDta3 = flghDta2.convert_dtypes()
display(flghDta3.head(5))

,apt_name_dp,apt_IATA_dp,apt_time_dt_dp,apt_name_ds,apt_IATA_ds,apt_time_dt_ds,price,ticket_sold_by
0,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,DUB,2026-03-12,231,
1,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,DUB,2026-03-12,243,
2,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,DUB,2026-03-12,361,
3,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,DUB,2026-03-12,433,
4,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,DUB,2026-03-12,603,


In [36]:
flghDta3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10773 entries, 0 to 10772
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   apt_name_dp     10773 non-null  string
 1   apt_IATA_dp     10773 non-null  string
 2   apt_time_dt_dp  10773 non-null  object
 3   apt_name_ds     10773 non-null  string
 4   apt_IATA_ds     10773 non-null  string
 5   apt_time_dt_ds  10773 non-null  object
 6   price           10773 non-null  Int64 
 7   ticket_sold_by  10773 non-null  string
dtypes: Int64(1), object(2), string(5)
memory usage: 684.0+ KB


In [37]:
# Inserting the Country and City columns
flghDta3.insert(4,'City_dp',None)
flghDta3.insert(4,'Country_dp',None)
flghDta3.insert(10,'City_ds',None)
flghDta3.insert(11,'Country_ds',None)
flghDta3.head(3)

,apt_name_dp,apt_IATA_dp,apt_time_dt_dp,apt_name_ds,Country_dp,City_dp,apt_IATA_ds,apt_time_dt_ds,price,ticket_sold_by,City_ds,Country_ds
0,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,None,None,DUB,2026-03-12,231,,None,None
1,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,None,None,DUB,2026-03-12,243,,None,None
2,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,None,None,DUB,2026-03-12,361,,None,None


In [38]:
# Addng the Country and City from the major_international_airport dataset for each flight path 
flghDtacpy = flghDta3.copy()
for idx,d in airports.iterrows():
    code = d['IATA_Code']
    Country = d['Country']
    City = d['City']
    flghDtacpy.loc[flghDta3['apt_IATA_dp'] == code,'City_dp'] = City
    flghDtacpy.loc[flghDta3['apt_IATA_dp'] == code,'Country_dp'] = Country
    flghDtacpy.loc[flghDta3['apt_IATA_ds'] == code,'City_ds'] = City
    flghDtacpy.loc[flghDta3['apt_IATA_ds'] == code,'Country_ds'] = Country
flghDtacpy.isnull().any()

apt_name_dp       False
apt_IATA_dp       False
apt_time_dt_dp    False
apt_name_ds       False
Country_dp        False
City_dp           False
apt_IATA_ds       False
apt_time_dt_ds    False
price             False
ticket_sold_by    False
City_ds           False
Country_ds        False
dtype: bool

In [39]:
flghDtacpy.head(10)

,apt_name_dp,apt_IATA_dp,apt_time_dt_dp,apt_name_ds,Country_dp,City_dp,apt_IATA_ds,apt_time_dt_ds,price,ticket_sold_by,City_ds,Country_ds
0,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,Netherlands,Amsterdam,DUB,2026-03-12,231,,Dublin,Ireland
1,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,Netherlands,Amsterdam,DUB,2026-03-12,243,,Dublin,Ireland
2,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,Netherlands,Amsterdam,DUB,2026-03-12,361,,Dublin,Ireland
3,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,Netherlands,Amsterdam,DUB,2026-03-12,433,,Dublin,Ireland
4,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,Netherlands,Amsterdam,DUB,2026-03-12,603,,Dublin,Ireland
5,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,Netherlands,Amsterdam,DUB,2026-03-12,616,,Dublin,Ireland
6,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,Netherlands,Amsterdam,DUB,2026-03-12,685,,Dublin,Ireland
7,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,Netherlands,Amsterdam,DUB,2026-03-12,723,,Dublin,Ireland
8,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,Netherlands,Amsterdam,DUB,2026-03-12,798,,Dublin,Ireland
9,Amsterdam Airport Schiphol,AMS,2026-03-14,Dublin Airport,Netherlands,Amsterdam,DUB,2026-03-14,191,,Dublin,Ireland


In [40]:
output_path = "flight_paths.csv"
flghDtacpy.to_csv(output_path, index=False)

In [41]:
flghDtacpy.shape

(10773, 12)

In [ ]:
# this fixes an error with one column having '-' instead of ['-'] for flight provider
f = pd.read_csv("flight_paths.csv")
f.head(2)

,apt_name_dp,apt_IATA_dp,apt_time_dt_dp,apt_name_ds,Country_dp,City_dp,apt_IATA_ds,apt_time_dt_ds,price,ticket_sold_by,City_ds,Country_ds
0,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,Netherlands,Amsterdam,DUB,2026-03-12,231,NaN,Dublin,Ireland
1,Amsterdam Airport Schiphol,AMS,2026-03-12,Dublin Airport,Netherlands,Amsterdam,DUB,2026-03-12,243,NaN,Dublin,Ireland


In [55]:
f['ticket_sold_by'] = f['ticket_sold_by'].fillna('-')

In [56]:
f['ticket_sold_by'] = f['ticket_sold_by'].apply(lambda x : [x])

In [57]:
f.head(2),f.tail(2)

(                  apt_name_dp apt_IATA_dp apt_time_dt_dp     apt_name_ds  \
 0  Amsterdam Airport Schiphol         AMS     2026-03-12  Dublin Airport   
 1  Amsterdam Airport Schiphol         AMS     2026-03-12  Dublin Airport   
 
     Country_dp    City_dp apt_IATA_ds apt_time_dt_ds  price ticket_sold_by  \
 0  Netherlands  Amsterdam         DUB     2026-03-12    231            [-]   
 1  Netherlands  Amsterdam         DUB     2026-03-12    243            [-]   
 
   City_ds Country_ds  
 0  Dublin    Ireland  
 1  Dublin    Ireland  ,
                                  apt_name_dp apt_IATA_dp apt_time_dt_dp  \
 10771  Toronto Pearson International Airport         YYZ     2026-12-05   
 10772  Toronto Pearson International Airport         YYZ     2026-12-07   
 
           apt_name_ds Country_dp  City_dp apt_IATA_ds apt_time_dt_ds  price  \
 10771  Dublin Airport     Canada  Toronto         DUB     2026-12-06   1130   
 10772  Dublin Airport     Canada  Toronto         DUB     2026-1

In [58]:
output_path = "flight_paths.csv"
f.to_csv(output_path, index=False)